In [ ]:
import json
import urllib.request
from datetime import datetime

from loguru import logger
from plant_care.config import get_env, load_config
from pyspark.sql import SparkSession

In [4]:
spark = SparkSession.builder.getOrCreate()

# Load config
env = get_env()
cfg = load_config("../project_config.yml", env)

In [14]:
# Fetch plant care data
# Source: PlantSolve API (CC BY 4.0, https://www.plantsolve.com). Requires a backlink when used.
BASE_URL = "https://www.plantsolve.com/api/v1/plants"
TIMEOUT_SECONDS = 30


def _section(data: dict, key: str) -> dict:
    """
    Get a nested section of a profile, defaulting to an empty dict.

    Args:
        data: Dictionary containing the section
        key: Section name

    Returns:
        The section dictionary, or an empty dict if it is missing or null
    """
    return data.get(key) or {}


def _get_json(url: str) -> dict | list:
    """
    Fetch a URL and decode its JSON body.

    Args:
        url: URL of the JSON resource

    Returns:
        Decoded JSON object or array
    """
    with urllib.request.urlopen(url, timeout=TIMEOUT_SECONDS) as response:
        return json.loads(response.read().decode("utf-8"))


def _recommended(parameters: dict, key: str) -> str | None:
    """
    Get the recommended value of a care parameter.

    Args:
        parameters: Parameters section of a plant profile
        key: Parameter name, such as temperature, humidity, or soilPH

    Returns:
        Recommended value, or None if the parameter is missing
    """
    return _section(parameters, key).get("recommended")


def parse_plant_profile(profile: dict) -> dict:
    """
    Flatten a PlantSolve plant profile into structured care fields.

    Args:
        profile: Raw plant profile from the PlantSolve detail endpoint

    Returns:
        Flat dictionary of plant care fields
    """
    care = _section(profile, "care")
    taxonomy = _section(profile, "taxonomy")
    lighting = _section(profile, "lighting")
    parameters = _section(profile, "parameters")
    nutrients = _section(profile, "nutrients")
    growth = _section(profile, "growthCharacteristics")
    pet_safe = _section(profile, "petSafe")

    return {
        "plant_id": profile.get("id"),
        "slug": profile.get("slug"),
        "common_name": profile.get("commonName"),
        "scientific_name": taxonomy.get("sciName"),
        "family": taxonomy.get("family"),
        "category": profile.get("category"),
        "tags": ", ".join(profile.get("taxonomyTags") or []),
        "summary": profile.get("summary"),
        "difficulty": care.get("difficulty"),
        "placement": care.get("placement"),
        "watering": care.get("watering"),
        "soil": care.get("soil"),
        "toxicity": care.get("toxicity"),
        "light_description": lighting.get("description"),
        "sunlight_hours": lighting.get("sunlightHours"),
        "light_intensity": lighting.get("intensity"),
        "temperature_recommended": _recommended(parameters, "temperature"),
        "humidity_recommended": _recommended(parameters, "humidity"),
        "soil_ph_recommended": _recommended(parameters, "soilPH"),
        "fertilizer_frequency": nutrients.get("fertilizerFrequency"),
        "growth_rate": growth.get("growthRate"),
        "mature_height": growth.get("matureHeight"),
        "pet_safe_cat": pet_safe.get("cat"),
        "pet_safe_dog": pet_safe.get("dog"),
        "url": profile.get("canonicalUrl"),
        "published_at": profile.get("publishedAt"),
        "updated_at": profile.get("updatedAt"),
        "ingestion_timestamp": datetime.now().isoformat(),
    }


def fetch_plant_care() -> list[dict]:
    """
    Fetch all plant care profiles from the PlantSolve API.

    Returns:
        List of structured plant care dictionaries from parse_plant_profile
    """
    index = _get_json(f"{BASE_URL}/index.json")
    logger.info(f"Found {len(index)} plants in PlantSolve index")

    plants = []
    for entry in index:
        slug = entry["slug"]
        try:
            profile = _get_json(f"{BASE_URL}/{slug}.json")
        except Exception as e:
            logger.warning(f"Failed to fetch plant '{slug}': {e}")
            continue
        plants.append(parse_plant_profile(profile))

    return plants


logger.info("Fetching plant care data...")
plants = fetch_plant_care()
logger.info(f"Fetched {len(plants)} plant profiles")
if plants:
    logger.info(f"Sample plant: {plants[0]['common_name']} ({plants[0]['scientific_name']})")

2026-11-10 22:16:16.594 | INFO     | __main__:<module>:122 - Fetching plant care data...


2026-11-10 22:16:16.962 | INFO     | __main__:fetch_plant_care:107 - Found 143 plants in PlantSolve index
2026-11-10 22:16:49.836 | INFO     | __main__:<module>:124 - Fetched 143 plant profiles
2026-11-10 22:16:49.837 | INFO     | __main__:<module>:126 - Sample plant: African Violet (Streptocarpus sect. Saintpaulia)


In [17]:
from pyspark.sql.types import BooleanType, StringType, StructField, StructType

NON_NULLABLE_FIELDS = {"plant_id", "slug", "common_name"}
BOOLEAN_FIELDS = {"pet_safe_cat", "pet_safe_dog"}
FIELD_NAMES = [
    "plant_id",
    "slug",
    "common_name",
    "scientific_name",
    "family",
    "category",
    "tags",
    "summary",
    "difficulty",
    "placement",
    "watering",
    "soil",
    "toxicity",
    "light_description",
    "sunlight_hours",
    "light_intensity",
    "temperature_recommended",
    "humidity_recommended",
    "soil_ph_recommended",
    "fertilizer_frequency",
    "growth_rate",
    "mature_height",
    "pet_safe_cat",
    "pet_safe_dog",
    "url",
    "published_at",
    "updated_at",
    "ingestion_timestamp",
]

schema = StructType(
    [
        StructField(
            name,
            BooleanType() if name in BOOLEAN_FIELDS else StringType(),
            name not in NON_NULLABLE_FIELDS,
        )
        for name in FIELD_NAMES
    ]
)

df = spark.createDataFrame(plants, schema=schema)
logger.info(f"Created DataFrame with {df.count()} plant records")
display(df)

2026-11-10 22:22:04.960 | INFO     | __main__:<module>:48 - Created DataFrame with 143 plant records


,plant_id,slug,common_name,scientific_name,family,category,tags,summary,difficulty,placement,watering,soil,toxicity,light_description,sunlight_hours,light_intensity,temperature_recommended,humidity_recommended,soil_ph_recommended,fertilizer_frequency,growth_rate,mature_height,pet_safe_cat,pet_safe_dog,url,published_at,updated_at,ingestion_timestamp
0,plant-african-violet,african-violet,African Violet,Streptocarpus sect. Saintpaulia,Gesneriaceae,Indoor Plants,"indoor-plants, pet-safe, flowering-plants, small-plants, moderate-care","Saintpaulia cultivars are compact, continuous-blooming perennials prized for their velvety foliage and colorful blossoms. This botanical guide outlines bottom-watering techniques to prevent foliar spotting, airy peat-perlite mixes, and bright indirect lighting needs.",intermediate,Bright indirect light on an East or North facing windowsill.,Bottom-water only when the top inch of soil is dry. Never get water on the fuzzy leaves.,"A very light, airy, peat-free mix (or a dedicated African Violet mix).",non-toxic,"Requires bright, indirect light to bloom continuously. Direct sun will burn the leaves, but too little light will stop flower production completely.",4-6 hours indirect,moderate,65°F - 80°F (18°C - 27°C),50% - 70%,6.0 - 6.5,Every two weeks during spring and summer (or quarter strength at every watering).,moderate,6-12 inches indoors,True,True,https://www.plantsolve.com/plants/african-violet/,2026-06-01,2026-07-20,2026-11-10T22:16:17.174745
1,plant-aglaonema-red-siam,aglaonema-red-siam,Aglaonema Red Siam,Aglaonema 'Siam Aurora',Araceae,Indoor Plants,"indoor-plants, low-light-tolerant, toxic-to-pets, statement-foliage, easy-care","Aglaonema 'Red Siam' is a highly resilient ornamental aroid featuring dramatic red-pink margins and bright green variegation. Learn light positioning to sustain intense coloration, substrate drainage requirements, and basic pest prevention.",beginner,"Highly adaptable. Survives in low light, but needs bright, indirect light to keep its neon red borders.",Water only when the top 2-3 inches of soil are dry. Very forgiving of missed waterings.,"A standard, well-draining indoor potting mix.",toxic,"While older green Aglaonemas can survive in pitch darkness, red varieties like the Siam Aurora require bright, indirect light to maintain their intense red borders.",4-6 hours indirect,medium,65°F - 85°F (18°C - 29°C),40% - 60%,5.5 - 6.5,Once every two months during spring and summer with a diluted liquid fertilizer.,slow,1-2 feet indoors,False,False,https://www.plantsolve.com/plants/aglaonema-red-siam/,2026-06-01,2026-07-20,2026-11-10T22:16:17.389748
2,plant-alocasia-amazonica,alocasia-amazonica,Alocasia Amazonica,Alocasia x amazonica (commonly sold as 'Polly'),Araceae,Indoor Plants,"indoor-plants, humidity-loving, toxic-to-pets, high-maintenance, statement-foliage","Alocasia × amazonica is a striking hybrid featuring dark green, arrowhead-shaped leaves with prominent white veins and scalloped edges. Master high-humidity management, aerated aroid soils, and seasonal dormancy care.",hard,"Bright, indirect light in a very warm, humid room. Keep far away from cold windows and drafts.",Keep the soil slightly moist but never waterlogged. Extremely susceptible to root rot.,"A chunky, highly aerated aroid mix. Heavy peat soils will kill it.",toxic,"Requires bright, indirect light to maintain its striking leaf coloration. Direct sun will severely scorch the leaves, while low light will cause it to drop leaves and go dormant.",4-8 hours indirect,bright,65°F - 85°F (18°C - 29°C),60% - 80%,5.5 - 6.5,Every 2 weeks during spring and summer with a balanced liquid fertilizer at half strength.,moderate,1-2 feet indoors,False,False,https://www.plantsolve.com/plants/alocasia-amazonica/,2026-06-01,2026-09-13,2026-11-10T22:16:17.600760
3,plant-alocasia-black-velvet,alocasia-black-velvet,Alocasia Black Velvet,Alocasia reginula,Araceae,Indoor Plants,"indoor-plants, high-humidity, toxic-to-pets, state